# Filter PrePPI by scores for 15k samples

## Load IXN file

In [ ]:
import pandas as pd

In [ ]:
df_ppi = pd.read_csv('../../GraPPI_data/ppi_axin.csv',
                         names=['p1','p2','dom1','dom2','score','template'],
                         header=None, delim_whitespace=True,
                         dtype={'p1':'string','p2':'string','ppi_id1':'string','ppi_id2':'string','score':'float16','template':'string'},
                         usecols=['dom1','dom2','score','template'],
                         memory_map=True,
                         na_filter=False
                         )

In [ ]:
df_ppi

## Check IXN stats

In [ ]:
df_ppi['score'].describe()

In [ ]:
# count how manny Nan in score
df_ppi['score'].min(), df_ppi['score'].max()

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df_ppi['score'], bins=200, range=(5,10))

In [ ]:
# sort by score and get the top 15000 negative samples
#df_ppi_sorted = df_ppi.sort_values(by='score', ascending=True)
#df_negative_samples = df_ppi_sorted.head(15000)
#df_negative_samples
# sort by score and get the sample with score between 2-10
df_ppi_sorted = df_ppi.sort_values(by='score', ascending=True)
df_negative_samples = df_ppi_sorted[(df_ppi_sorted['score'] >= 5) & (df_ppi_sorted['score'] <= 10)]
#df_negative_samples.to_csv('../../GraPPI_data/negative_samples.csv', index=False)
df_negative_samples

In [ ]:
df_negative_samples.describe()

## Get filtered samples from IXN file

In [ ]:
df_tpl = df_negative_samples['template'].str.split(':', n=2, expand=True)
df_negative_samples['pdb_id'] = df_tpl[0]
df_negative_samples['chain_order'] = df_tpl[1]
# drop template column
df_negative_samples = df_negative_samples.drop(columns=['template'])
df_negative_samples.head()

In [ ]:
df_negative_samples.to_csv('../../GraPPI_data/ppi_axin_negative_df.csv', index=False,na_rep='')
#df_negative_samples.to_csv('../../GraPPI_data/ppi_axin_negative_df.csv', index=False)

## Check saved dataframe and remove experimental samples

In [ ]:
import pandas as pd

In [ ]:
df_negative_samples = pd.read_csv('../../GraPPI_data/ppi_axin_negative_df.csv',
            keep_default_na=False,
            na_values=[])

In [ ]:
df_negative_samples

In [ ]:
map_dict = {line.strip().split('\t>')[0]: set(line.strip().split('\t>')[1:])
    for line in open("/mnt/poisson/data/shares/PrePPI_PPIs/Human_af/map_list")}
expDB_set = [set(line.strip().split('_'))
             for line in open("/mnt/poisson/data/shares/PrePPI_PPIs/Human_af/human_allExpDB.csv")][1:]
expDB_lookup = {frozenset(x) for x in expDB_set}
# remove rows from df where the pair is in expDB_lookup
df_filtered = df_negative_samples[~df_negative_samples.apply(lambda row: any(frozenset((a, b)) in expDB_lookup
            for a in map_dict.get(row['dom1'].split('.')[0], []) for b in map_dict.get(row['dom2'].split('.')[0], [])), axis=1)]
df_filtered.to_csv('../../GraPPI_data/ppi_axin_negative_df.csv', index=False,na_rep='')


In [ ]:
df_filtered

# Run code to get aligned pdbs

```bash
python IXN2Pdb.py --IXN_csv_file ../../GraPPI_data/ppi_axin_negative_df.csv --base_dir /mnt/poisson/data/ssnegi/tmp_dir/Human1/Seqs/ --aligned_dir ../../GraPPI_data/preppi_aligned/
```

# Get preppi table with chain ids

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

In [ ]:
pdb_list = [fn.replace('.pdb', '') for fn in os.listdir('../../GraPPI_data/preppi_aligned/') if fn.endswith('.pdb')]

In [ ]:
len(pdb_list)

In [ ]:
# Initialize the DataFrame
df = pd.DataFrame(columns=['PDB', 'Receptor Chains', 'Ligand Chains', 'paths', 'affinity'])
# Populate the DataFrame
for pdb_id in tqdm(pdb_list):
    pdb_path = f'../../GraPPI_data/preppi_aligned/{pdb_id}.pdb'
    receptor_chain, ligand_chain = 'A', 'B'  # Placeholder for actual chain extraction logic
    new_row = {
		'PDB': pdb_id,
		'Receptor Chains': receptor_chain,
		'Ligand Chains': ligand_chain,
		'paths': pdb_path,
		'affinity': -1
	}
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

df

In [ ]:
df.to_csv('./saved_tables/preppi_db.csv', index=False)